In [16]:
from datasets import load_dataset
from transformers import AutoModel,AutoTokenizer
import torch
from qdrant_client import QdrantClient
from qdrant_client.http import models
from qdrant_client.http.models import CollectionStatus

In [2]:
dataset=load_dataset("ag_news",split="train")
dataset

Generating test split: 100%|██████████| 7600/7600 [00:00<00:00, 331720.80 examples/s]


Dataset({
    features: ['text', 'label'],
    num_rows: 120000
})

In [3]:
from random import choice
for i in range(5):
    random_sample=choice(range(len(dataset)))
    print("*"*50)
    print(dataset[random_sample]['text'])

**************************************************
Larkin, Vizquel and Garciaparra File for Free Agency (Reuters) Reuters - Two of the greatest shortstops\ever, Barry Larkin and Omar Vizquel, were among the 58 players\to file for free agency on Friday.
**************************************************
EDS expected to slash 20,000 jobs Electronic Data Systems (EDS), one of the world #39;s biggest computer services companies, is likely to cut up to 20,000 jobs over the next two years, the company #39;s chief executive said.
**************************************************
Google Grabs 3-D Mapping Startup Google Inc. on Wednesday branched out into 3-D digital mapping with the acquisition of Keyhole Corp. In its second acquisition this year, Google has bought a startup company that connects consumers, businesses 
**************************************************
Showjumper Faces Losing Gold Medal Irish showjumper Cian OConnor was facing the prospect of losing his Olympics gold medal to

In [5]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer=AutoTokenizer.from_pretrained('gpt2')
model=AutoModel.from_pretrained('gpt2').to(device)

/home/dl-user/mario/sampleEnv/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12060). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [7]:
tokenizer.pad_token=tokenizer.eos_token

In [8]:
tokenizer.vocab_size

50257

In [9]:
#We need one vector to represent a single sentence hence mean pooling

def mean_pooling(model_output, attention_mask):

    token_embeddings = model_output[0]
    input_mask_expanded = (attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float())
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
    sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    return sum_embeddings / sum_mask


In [10]:
def embed_text(examples):
    inputs = tokenizer(
        examples["text"], padding=True, truncation=True, return_tensors="pt"
    )#.to(device)
    with torch.no_grad():
        model_output = model(**inputs)
    pooled_embeds = mean_pooling(model_output, inputs["attention_mask"])
    return {"embedding": pooled_embeds.cpu().numpy()}

In [11]:
small_set = (
    dataset.shuffle(42) # randomly shuffles the data, 42 is the seed
           .select(range(1000)) # we'll take 1k rows
           .map(embed_text, batched=True, batch_size=128) # and apply our function above to 128 articles at a time
)

Map: 100%|██████████| 1000/1000 [01:13<00:00, 13.61 examples/s]


In [12]:
small_set

Dataset({
    features: ['text', 'label', 'embedding'],
    num_rows: 1000
})

In [13]:
n_rows = range(len(small_set))
small_set = small_set.add_column("idx", n_rows)
small_set

Dataset({
    features: ['text', 'label', 'embedding', 'idx'],
    num_rows: 1000
})

In [14]:
id2label = {str(i): label for i, label in enumerate(dataset.features["label"].names)}

In [15]:
def get_names(label_num):
    return id2label[str(label_num)]

label_names = list(map(get_names, small_set['label']))
small_set = small_set.add_column("label_names", label_names)
small_set

Dataset({
    features: ['text', 'label', 'embedding', 'idx', 'label_names'],
    num_rows: 1000
})

In [18]:
dim_size = len(small_set[0]["embedding"]) 
dim_size

768

In [17]:
client = QdrantClient(host="localhost", port=6370)
client

In [19]:
my_collection = "news_embeddings"
client.recreate_collection(
    collection_name=my_collection,
    vectors_config=models.VectorParams(size=dim_size, distance=models.Distance.COSINE)
)

/tmp/ipykernel_4156653/2527313622.py:2: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

In [20]:
payloads = small_set.select_columns(["label_names", "text"]).to_pandas().to_dict(orient="records")

In [21]:
client.upsert(
    collection_name=my_collection,
    points=models.Batch(
        ids=small_set["idx"],
        vectors=small_set["embedding"],
        payloads=payloads
    )
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [22]:
# Step 1 - Select Random Sample
query2 = {"text": dataset[choice(range(len(dataset)))]['text']}
query2

{'text': 'Price of Average U.S. House Rises 8.5 Pct  WASHINGTON (Reuters) - The October average U.S. house price  jumped 8.5 percent from a year ago, setting the stage for  mortgage giants Fannie Mae and Freddie Mac to raise the limit  on mortgages they can buy, a Federal Housing Finance Board  survey showed on Tuesday.'}

In [23]:
# Step 2 - Create a Vector
query2 = embed_text(query2)['embedding'][0, :]
query2.shape, query2[:20]

((768,),
 array([ 0.21156988, -0.33831218,  0.14692633,  0.16530581,  0.11433054,
        -0.03272829,  7.0509357 ,  0.5372854 ,  0.30431715,  0.12386503,
        -0.0460727 , -0.30912313, -0.13335751, -0.03603826, -0.3617902 ,
         0.3452387 , -0.23940775, -0.2841687 ,  0.14975855, -0.782955  ],
       dtype=float32))

In [29]:
# Step 3 - Search for similar articles. Don't forget to convert the vector to a list.
client.query_points(
    collection_name=my_collection,
    query=query2.tolist(),
    limit=5
)

QueryResponse(points=[ScoredPoint(id=98, version=1, score=0.99945605, payload={'label_names': 'Business', 'text': 'U.S. Stocks, Dollar and Bonds Fall  NEW YORK (Reuters) - U.S. stocks and bonds fell on Tuesday  as the dollar hit a record low against the euro.'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=582, version=1, score=0.999279, payload={'label_names': 'Business', 'text': 'US Treasuries Cut Losses on Job Woes Hint  NEW YORK (Reuters) - Treasuries trimmed early losses on  Monday as evidence of sluggish hiring by U.S. manufacturers   suggested little need for the Federal Reserve to be more  aggressive in hiking interest rates.'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=518, version=1, score=0.9991644, payload={'label_names': 'Business', 'text': 'Treasuries Pressured by Oil, Upbeat Fed  NEW YORK (Reuters) - U.S. Treasuries prices eased for a  third straight session on Tuesday, unnerved by a mix of softer  oil, firmer stocks and an upbeat e